<a href="https://colab.research.google.com/github/ancestor9/Special_notes/blob/main/gradio_twinkle_twinkle_little_star.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Gradio Generate Tone**

In [ ]:
import numpy as np
import gradio as gr

notes = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
note_map = {n: i for i, n in enumerate(notes)}

# 단일 음 생성
def generate_tone(note_index, octave, duration):
    sr = 48000
    a4_freq = 440
    tones_from_a4 = 12 * (octave - 4) + (note_index - 9)
    frequency = a4_freq * 2 ** (tones_from_a4 / 12)
    t = np.linspace(0, duration, int(duration * sr))

    # ADSR 엔벨로프 적용 (자연스러운 소리를 위해)
    attack_time = 0.05
    decay_time = 0.1
    sustain_level = 0.7
    release_time = 0.1

    attack_samples = int(attack_time * sr)
    decay_samples = int(decay_time * sr)
    release_samples = int(release_time * sr)
    sustain_samples = len(t) - attack_samples - decay_samples - release_samples

    envelope = np.ones(len(t))

    if len(t) > attack_samples + decay_samples + release_samples:
        # Attack
        envelope[:attack_samples] = np.linspace(0, 1, attack_samples)
        # Decay
        envelope[attack_samples:attack_samples + decay_samples] = np.linspace(1, sustain_level, decay_samples)
        # Sustain
        envelope[attack_samples + decay_samples:attack_samples + decay_samples + sustain_samples] = sustain_level
        # Release
        envelope[-release_samples:] = np.linspace(sustain_level, 0, release_samples)

    # 기본 사인파에 약간의 하모닉 추가 (더 풍부한 소리)
    audio = np.sin(t * 2 * np.pi * frequency)
    audio += 0.3 * np.sin(t * 2 * np.pi * frequency * 2)  # 2차 하모닉
    audio += 0.1 * np.sin(t * 2 * np.pi * frequency * 3)  # 3차 하모닉

    audio = audio * envelope
    audio = (audio * 15000).astype(np.int16)

    return sr, audio

# 동요 멜로디 생성
def generate_song():
    sr = 48000
    # "반짝반짝 작은별" 멜로디
    melody = [
        ("C", 4, 0.5), ("C", 4, 0.5), ("G", 4, 0.5), ("G", 4, 0.5),
        ("A", 4, 0.5), ("A", 4, 0.5), ("G", 4, 1.0),
        ("F", 4, 0.5), ("F", 4, 0.5), ("E", 4, 0.5), ("E", 4, 0.5),
        ("D", 4, 0.5), ("D", 4, 0.5), ("C", 4, 1.0)
    ]

    song = np.array([], dtype=np.int16)

    for note, octave, duration in melody:
        note_index = note_map[note]
        _, tone = generate_tone(note_index, octave, duration)
        # 음표 사이에 작은 간격 추가
        pause = np.zeros(int(0.05 * sr), dtype=np.int16)
        song = np.concatenate((song, tone, pause))

    return sr, song

# 사용자 정의 멜로디 생성
def generate_custom_song(note_sequence, octave, note_duration):
    sr = 48000
    if not note_sequence.strip():
        return generate_song()  # 빈 입력시 기본 노래 재생

    # 입력된 음표들을 파싱 (예: "C D E F G A B")
    note_list = note_sequence.upper().replace(',', ' ').split()

    song = np.array([], dtype=np.int16)

    for note in note_list:
        if note in note_map:
            note_index = note_map[note]
            _, tone = generate_tone(note_index, octave, note_duration)
            pause = np.zeros(int(0.05 * sr), dtype=np.int16)
            song = np.concatenate((song, tone, pause))

    if len(song) == 0:  # 유효한 음표가 없으면 기본 노래
        return generate_song()

    return sr, song

# Gradio 인터페이스
with gr.Blocks(title="🎶 음악 생성기", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎶 반짝반짝 작은별 & 음악 생성기")
    gr.Markdown("기본 노래를 재생하거나 원하는 멜로디를 만들어보세요!")

    with gr.Tab("기본 노래"):
        gr.Markdown("### 반짝반짝 작은별을 연주합니다")
        play_button = gr.Button("🎵 노래 재생", variant="primary", size="lg")
        audio_output1 = gr.Audio(label="생성된 음악")

        play_button.click(
            fn=generate_song,
            outputs=audio_output1
        )

    with gr.Tab("사용자 정의 멜로디"):
        gr.Markdown("### 원하는 멜로디를 만들어보세요")
        gr.Markdown("음표를 공백으로 구분해서 입력하세요 (예: C D E F G A B)")

        with gr.Row():
            note_input = gr.Textbox(
                label="음표 시퀀스",
                placeholder="C C G G A A G F F E E D D C",
                value="C C G G A A G F F E E D D C"
            )

        with gr.Row():
            octave_slider = gr.Slider(
                minimum=3, maximum=6, value=4, step=1,
                label="옥타브"
            )
            duration_slider = gr.Slider(
                minimum=0.2, maximum=2.0, value=0.5, step=0.1,
                label="음표 길이 (초)"
            )

        create_button = gr.Button("🎼 멜로디 생성", variant="primary")
        audio_output2 = gr.Audio(label="생성된 멜로디")

        create_button.click(
            fn=generate_custom_song,
            inputs=[note_input, octave_slider, duration_slider],
            outputs=audio_output2
        )

    with gr.Tab("사용 가능한 음표"):
        gr.Markdown("### 사용 가능한 음표들")
        gr.Markdown("**" + " | ".join(notes) + "**")
        gr.Markdown("""
        - **C, D, E, F, G, A, B**: 자연음
        - **C#, D#, F#, G#, A#**: 반음 (샤프)
        - 음표들을 공백이나 쉼표로 구분해서 입력하세요
        - 예시: `C D E F G` 또는 `C, D, E, F, G`
        """)

if __name__ == "__main__":
    demo.launch()


/tmp/ipykernel_2105/3543682330.py:94: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="🎶 음악 생성기", theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3ef8999a1759aa7480.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import numpy as np
from scipy.io import wavfile

# 1. 음계별 주파수 정의 (4옥타브 기준)
NOTE_FREQ = {
    '도': 261.63,
    '레': 293.66,
    '미': 329.63,
    '파': 349.23,
    '솔': 392.00,
    '라': 440.00,
    '시': 493.88,
    '쉬': 0.0      # 쉼표
}

# 2. '반짝반짝 작은별' 멜로디와 박자 구성
# (음계, 박자수) -> 1박자를 0.5초로 계산할 예정
MELODY = [
    ('솔', 1), ('솔', 1), ('라', 1), ('라', 1), ('솔', 1), ('솔', 1), ('파', 2),
    ('미', 1), ('미', 1), ('레', 1), ('레', 1), ('도', 2),
    ('솔', 1), ('솔', 1), ('파', 1), ('파', 1), ('미', 1), ('미', 1), ('레', 2),
    ('솔', 1), ('솔', 1), ('파', 1), ('파', 1), ('미', 1), ('미', 1), ('레', 2),
    ('도', 1), ('도', 1), ('솔', 1), ('솔', 1), ('라', 1), ('라', 1), ('솔', 2),
    ('파', 1), ('파', 1), ('미', 1), ('미', 1), ('레', 1), ('레', 1), ('도', 2)
]

def generate_twinkle_star(bpm):
    """
    사용자가 입력한 BPM에 맞춰 '반짝반짝 작은별' 오디오 데이터를 생성합니다.
    """
    sample_rate = 44100  # CD 음질 샘플링 레이트
    beat_duration = 60.0 / bpm  # 1박자당 시간(초)

    audio_buffer = []

    for note, beat in MELODY:
        duration = beat_duration * beat
        t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

        if note == '쉬':
            # 쉼표인 경우 무음 처리
            wave = np.zeros_like(t)
        else:
            freq = NOTE_FREQ[note]
            # 기본 사인파 생성
            wave = np.sin(2 * np.pi * freq * t)

            # 딱딱한 기계음을 줄이기 위한 간단한 페이드 인/아웃(Envelope) 적용
            fade_len = int(sample_rate * 0.02)  # 0.02초 동안 페이드
            if len(wave) > fade_len * 2:
                envelope = np.ones_like(wave)
                envelope[:fade_len] = np.linspace(0, 1, fade_len)
                envelope[-fade_len:] = np.linspace(1, 0, fade_len)
                wave *= envelope

        audio_buffer.append(wave)

    # 모든 음을 하나로 합치기
    full_audio = np.concatenate(audio_buffer)

    # float32 데이터를 16-bit PCM WAV 형식 오디오로 변환 (-32768 ~ 32767)
    audio_int16 = (full_audio * 32767).astype(np.int16)

    # Gradio 오디오 컴포넌트는 (샘플레이트, 넘파이배열) 튜플을 반환하면 재생해 줍니다.
    return (sample_rate, audio_int16)

# 3. Gradio 인터페이스 구성
with gr.Blocks(title="반짝반짝 작은별 플레이어") as demo:
    gr.Markdown("## ⭐️ 파이썬으로 듣는 반짝반짝 작은별")
    gr.Markdown("템포(BPM)를 조절하고 버튼을 눌러 노래를 만들어보세요!")

    with gr.Row():
        bpm_slider = gr.Slider(
            minimum=60,
            maximum=200,
            value=120,
            step=5,
            label="연주 속도 (BPM)"
        )

    play_btn = gr.Button("🎵 노래 생성 및 재생", variant="primary")
    audio_output = gr.Audio(label="생성된 오디오", type="numpy")

    # 버튼 클릭 시 함수 연결
    play_btn.click(
        fn=generate_twinkle_star,
        inputs=[bpm_slider],
        outputs=[audio_output]
    )

# 4. 웹 서버 실행
if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bd957d4e8732d4f2bb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
